<a href="https://colab.research.google.com/github/matheus-cmc/atividades_IA_industrial/blob/main/ResolucaoAtividade01UC03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Análise e Explicação Detalhada do Gabarito

O documento apresenta uma solução completa e bem estruturada para treinar um modelo de classificação com XGBoost. Abaixo, detalhamos o funcionamento de cada etapa.

## 1. Objetivo da Atividade
O objetivo central é capacitar o aluno a realizar a exploração, preparação e treinamento de um modelo XGBoost. A prática abrange todo o fluxo, desde o download e carregamento dos dados até a execução do treinamento e a avaliação de sua performance final.

## 2. Estrutura do Desafio Prático
O desafio é dividido em quatro etapas principais, que representam o fluxo de trabalho padrão em um projeto de Machine Learning:


1. **Importação e Exploração dos Dados**: Carregar o dataset e realizar uma análise inicial para entender sua estrutura e conteúdo.


2. **Preparação e Pré-processamento**: Tratar os dados para que possam ser utilizados pelo modelo, o que inclui a separação de features e alvo, a conversão de dados categóricos e a divisão em conjuntos de treino e teste.


3. **Treinamento do Modelo XGBoost**: Instanciar e treinar o classificador usando os dados de treinamento.


4. **Avaliação do Modelo**: Utilizar os dados de teste para fazer predições e avaliar o desempenho do modelo com métricas como Acurácia, Relatório de Classificação e Matriz de Confusão.

In [ ]:
#Importações essenciais:

import os, glob #Utilitários para interagir com o sistema de arquivos. No código,
import pandas as pd #Fundamental para a manipulação de dados em formato de tabelas (DataFrames).

#scikit-learn a principal biblioteca de Machine Learning em Python.
from sklearn.model_selection import train_test_split #Para dividir os dados em conjuntos de treino e teste.
from sklearn.compose import ColumnTransformer #Para aplicar diferentes transformações a diferentes colunas (essencial para tratar dados numéricos e categóricos separadamente)
from sklearn.preprocessing import OneHotEncoder #Para converter variáveis categóricas (texto) em um formato numérico que o modelo entenda.
from sklearn.pipeline import Pipeline #Para encadear as etapas de pré-processamento e o modelo em um único fluxo de trabalho, garantindo consistência e evitando vazamento de dados (data leakage)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix #Métricas para avaliar a performance do modelo.
from xgboost import XGBClassifier #Contém a implementação do XGBClassifier, o modelo de Gradient Boosting que será treinado

In [ ]:
# ---------- Funções auxiliares ----------

#verificar se o nome do arquivo está correto, igual ao código abaixo
def CarregarArquivo(nome_arquivo="flights_delays_120.csv"):
    if os.path.exists(nome_arquivo):
        return pd.read_csv(nome_arquivo)
    else:
        raise FileNotFoundError(f"Arquivo {nome_arquivo} não encontrado.")


# Procura por nomes de coluna comuns para a variável alvo (como "delayed", "alvo", "target") e a retorna.
# Isso torna o script mais flexível, pois ele pode se adaptar a datasets com diferentes nomes para a coluna de destino.

def coluna_alvo(dados):
    candidatos = ["delayed", "atraso", "alvo", "target", "y"]
    for c in candidatos:
        if c in dados.columns:
            return c
    raise KeyError("Coluna alvo não encontrada.")

In [ ]:
CarregarArquivo()

,airline,origin,destination,departure_hour,day_of_week,weather,delayed
0,TravelAir,GIG,FOR,11,5,Storm,0
1,JetCloud,CNF,SSA,11,3,Wind,0
2,SkyWings,POA,SSA,4,5,Fog,1
3,JetCloud,BSB,FOR,6,4,Storm,1
4,JetCloud,GRU,FOR,3,1,Rain,1
...,...,...,...,...,...,...,...
115,JetCloud,GIG,FOR,8,5,Wind,0
116,FlyFast,GRU,CWB,6,4,Rain,1
117,SkyWings,CNF,BEL,3,4,Rain,1
118,AirOne,POA,SSA,20,6,Storm,1


In [ ]:
def main():
    # Carrega os dados do arquivo CSV para um DataFrame do pandas.
    dados = carregar_csv()

    # Identifica e armazena o nome da coluna que será o nosso alvo de previsão (ex: "delayed").
    alvo = coluna_alvo(dados)

    #Cria a variável 'y', que contém apenas a coluna alvo (os dados que queremos prever).
    y = dados[alvo]

    #Cria a variável 'X', que contém todas as outras colunas, ou seja, as nossas features (dados de entrada).
    X = dados.drop(columns=[alvo])

    #Cria uma lista com os nomes das colunas que são de texto (categóricas), identificadas pelo tipo 'object'.
    colunas_categoricas = [c for c in X.columns if X[c].dtype == "object"]

    #Cria uma lista com os nomes das colunas que não são de texto, ou seja, as colunas numéricas.
    colunas_numericas = [c for c in X.columns if c not in colunas_categoricas]

  # colunas_categoricas = X.select_dtypes(include='object').columns.to_list()
  # colunas_numericas = X.select_dtypes(exclude='object').columns.to_list()

   #Divide os dados em conjuntos de treino (80%) e teste (20%).

    # X: É o seu DataFrame contendo todas as features (as colunas de entrada que o modelo usará para aprender, como "airline", "flight", "airport_from", etc.).
    # y: É a sua Série (ou coluna) contendo o alvo (a variável que você quer prever, no caso, se o voo atrasou ou não).
    # test_size=0.2: significa que 20% de todos os seus dados serão separados para o conjunto de teste (X_teste, y_teste). Consequentemente, os 80% restantes serão usados para o conjunto de treino
    # 'random_state=42' garante que a divisão seja sempre a mesma, para reprodutibilidade.

    # 'stratify=y' mantém a proporção de classes (atrasos/não atrasos) igual nos dois conjuntos.

    X_treino, X_teste, y_treino, y_teste = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

        #Configura um transformador que aplicará passos diferentes para cada tipo de coluna.
    preprocessador = ColumnTransformer([
        #Para as colunas categóricas, aplica o OneHotEncoder que transforma texto em números (0s e 1s).
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), colunas_categoricas),
        #Para as colunas numéricas, 'passthrough' indica que elas não sofrerão nenhuma alteração.
        ("numericas", "passthrough", colunas_numericas),
    ])
 #Instancia o modelo de classificação XGBoost com hiperparâmetros específicos.
    #n_estimators: número de árvores de decisão.
    #learning_rate: a "velocidade" de aprendizado do modelo.
    #max_depth: a profundidade máxima de cada árvore.
    modelo = XGBClassifier(
        n_estimators=250, learning_rate=0.1,
        max_depth=6, subsample=0.9, colsample_bytree=0.9,
        eval_metric="logloss", random_state=42
    )
    #Cria um Pipeline que encadeia o pré-processamento e o modelo em um único fluxo.
    #Isso garante que os mesmos passos sejam aplicados tanto no treino quanto no teste.
    pipeline = Pipeline([
        ("prep", preprocessador),
        ("xgb", modelo)
    ])


    #Treina o pipeline completo (pré-processador + modelo) usando os dados de treino.
    pipeline.fit(X_treino, y_treino)


    #Faz as previsões (0 ou 1) para o conjunto de teste, que o modelo nunca viu antes.
    y_pred = pipeline.predict(X_teste)

    #Imprime a acurácia: a porcentagem de previsões corretas.
    print("Acurácia:", accuracy_score(y_teste, y_pred))


    #Imprime o relatório de classificação, com métricas detalhadas como precisão e recall para cada classe.
    print("Relatório:\n", classification_report(y_teste, y_pred))


    #Imprime a matriz de confusão, que mostra onde o modelo acertou e errou (Verdadeiros Positivos, Falsos Negativos, etc.).
    print("Matriz de confusão:\n", confusion_matrix(y_teste, y_pred))


In [ ]:
if __name__ == "__main__":
    main()

Acurácia: 0.875
Relatório:
               precision    recall  f1-score   support

           0       0.92      0.86      0.89        14
           1       0.82      0.90      0.86        10

    accuracy                           0.88        24
   macro avg       0.87      0.88      0.87        24
weighted avg       0.88      0.88      0.88        24

Matriz de confusão:
 [[12  2]
 [ 1  9]]
